# How Can a Wellness Technology Company Play It Smart?## Bellabeat Marketing Analysis — Google Data Analytics Capstone, Case Study 2**Mohammad Saeed Angiz**Every number in this notebook is calculated from the data at run time, not typed in by hand.

## Setup`repr.plot.*` is the Jupyter equivalent of RMarkdown's `fig.width` / `fig.height`.

In [ ]:
options(repr.plot.width = 9, repr.plot.height = 5.5, repr.plot.res = 120)options(warn = -1)if (!requireNamespace("janitor", quietly = TRUE)) install.packages("janitor")suppressPackageStartupMessages({  library(tidyverse)  library(lubridate)  library(janitor)  library(scales)  library(IRdisplay)})Sys.setlocale("LC_TIME", "C")   # English weekday names regardless of localecat("packages loaded\n")

## Locating the dataKaggle mounts datasets under `/kaggle/input`, a local machine does not. Rather thanhard-coding folder names, this searches for the directories that actually contain`dailyActivity_merged.csv`, so the same code runs in both places.The dataset ships as **two export folders** covering consecutive months. Most publishedanalyses use only the second. Both share an identical `dailyActivity_merged.csv` schema,so this analysis combines them — which roughly doubles the observation window.

In [ ]:
find_fitabase_dirs <- function(root) {  hits <- list.files(root, pattern = "^dailyActivity_merged\\.csv$",                     recursive = TRUE, full.names = TRUE, ignore.case = TRUE)  sort(unique(dirname(hits)))}ROOT <- if (dir.exists("/kaggle/input")) "/kaggle/input" else  "C:/Users/angiz/Desktop/FitBit Fitness Tracker Data"dirs <- find_fitabase_dirs(ROOT)cat("Searched under:", ROOT, "\n")for (d in dirs) cat("  found:", d, "\n")P1 <- dirs[grepl("3\\.12\\.16", dirs)][1]      # Mar 12 - Apr 11P2 <- dirs[grepl("4\\.12\\.16-5", dirs)][1]    # Apr 12 - May 12HAS_MARCH <- !is.na(P1)if (is.na(P2)) stop("Could not find the Apr 12 - May 12 export folder.")if (!HAS_MARCH) cat("\nNOTE: only one export folder present - March data unavailable.\n")

## Load and cleanCleaning decisions applied here:1. **Combined both export folders**, keeping a `period` column so every row stays traceable.2. **Standardised column names** to `snake_case` with `janitor::clean_names()`.3. **Parsed dates from text** — `read_csv()` imports `"4/12/2016"` as a character string.4. **Removed duplicate records** — `sleepDay_merged.csv` contains three exact duplicates.5. **Flagged non-wear days** rather than deleting them (see below).6. **Aggregated March minute-level sleep to nightly totals** using a 6 a.m. cutoff.

In [ ]:
read_period <- function(path, file) {  if (is.na(path)) return(NULL)  read_csv(file.path(path, file), show_col_types = FALSE)}# ---- Daily activity -----------------------------------------------------daily_raw <- bind_rows(  if (HAS_MARCH) read_period(P1, "dailyActivity_merged.csv") |>      mutate(period = "Mar 12 - Apr 11") else NULL,  read_period(P2, "dailyActivity_merged.csv") |>      mutate(period = "Apr 12 - May 12")) |> clean_names()daily <- daily_raw |>  mutate(date = mdy(activity_date)) |>  distinct(id, date, .keep_all = TRUE) |>  mutate(weekday     = wday(date, label = TRUE, abbr = FALSE),         is_wear_day = total_steps > 0)worn <- daily |> filter(is_wear_day)# ---- Sleep --------------------------------------------------------------sleep_apr_raw <- read_period(P2, "sleepDay_merged.csv") |> clean_names()sleep_apr <- sleep_apr_raw |>  mutate(date = as_date(mdy_hms(sleep_day))) |>  distinct(id, date, .keep_all = TRUE) |>  transmute(id, date,            minutes_asleep = total_minutes_asleep,            time_in_bed    = total_time_in_bed)sleep_mar <- if (HAS_MARCH) {  read_period(P1, "minuteSleep_merged.csv") |>    clean_names() |>    distinct(id, date, .keep_all = TRUE) |>    mutate(ts = mdy_hms(date), sleep_date = as_date(ts - hours(6))) |>    group_by(id, date = sleep_date) |>    summarise(minutes_asleep = sum(value == 1), time_in_bed = n(), .groups = "drop") |>    filter(time_in_bed >= 60)} else NULLsleep <- bind_rows(sleep_mar, sleep_apr) |>  distinct(id, date, .keep_all = TRUE) |>  mutate(hours_asleep     = minutes_asleep / 60,         awake_in_bed     = time_in_bed - minutes_asleep,         sleep_efficiency = minutes_asleep / time_in_bed)# ---- Hourly steps and weight -------------------------------------------hourly <- bind_rows(  if (HAS_MARCH) read_period(P1, "hourlySteps_merged.csv") else NULL,  read_period(P2, "hourlySteps_merged.csv")) |> clean_names() |>  mutate(ts = mdy_hms(activity_hour), hour = hour(ts)) |>  distinct(id, ts, .keep_all = TRUE)weight <- bind_rows(  if (HAS_MARCH) read_period(P1, "weightLogInfo_merged.csv") else NULL,  read_period(P2, "weightLogInfo_merged.csv")) |> clean_names() |>  mutate(date = as_date(mdy_hms(date))) |>  distinct(id, date, .keep_all = TRUE)# ---- Chart styling ------------------------------------------------------TEAL <- "#3E7C79"; MINT <- "#7FC8A9"; CORAL <- "#E8836F"SAND <- "#F2C57C"; SLATE <- "#5C6B73"; GREY <- "#D9DEE0"theme_bb <- theme_minimal(base_size = 13) +  theme(plot.title    = element_text(face = "bold", size = 15),        plot.subtitle = element_text(colour = SLATE, size = 11),        panel.grid.minor = element_blank())n_users <- n_distinct(daily$id)sa <- worn |> inner_join(sleep, by = c("id", "date"))cat("users:", n_users, "| worn days:", nrow(worn), "| sleep nights:", nrow(sleep), "\n")

In [ ]:
wear <- worn |> count(id, name = "d") |>  left_join(daily |> distinct(id, period) |>    left_join(daily |> group_by(period) |>      summarise(w = as.numeric(max(date) - min(date)) + 1, .groups = "drop"),      by = "period") |>    group_by(id) |> summarise(w = sum(w), .groups = "drop"), by = "id") |>  mutate(rate = d / w)display_markdown(sprintf("## Executive summaryTwo months of Fitbit tracker data from **%d users** shows the wellness market has beenselling the wrong habit. The strongest relationship in the data is not between steps andhealth — it is between **sitting still and sleeping badly**.| Metric | Value ||---|---|| Users analysed | %d || Days of tracked activity | %s || Nights of sleep | %s || Median device wear rate | %s || Average sedentary time per day | %s hours || Nights under 7 hours of sleep | %s || Days reaching 10,000 steps | %s || Correlation: sedentary time vs sleep | %s |**Top three recommendations:** replace the fixed 10,000-step goal with an adaptive one;market the Bellabeat app on the sit-less / sleep-better link; and time notifications tothe two hours when users are already moving.",  n_users, n_users, comma(nrow(worn)), comma(nrow(sleep)),  percent(median(wear$rate), 1), round(mean(worn$sedentary_minutes)/60, 1),  percent(mean(sleep$hours_asleep < 7), 0.1),  percent(mean(worn$total_steps >= 10000), 0.1),  round(cor(sa$sedentary_minutes, sa$hours_asleep), 2)))

---# 1. Ask — Summary of the business task> **Deliverable 1: A clear summary of the business task**Bellabeat is a high-tech manufacturer of health-focused products for women, founded in 2013by Urška Sršen and Sando Mur. The product line comprises the Bellabeat app, the Leaf tracker,the Time watch, the Spring water bottle and a subscription membership.Urška Sršen has asked the marketing analytics team to analyse usage data from**non-Bellabeat smart devices**, select **one Bellabeat product**, and produce high-levelrecommendations for marketing strategy.### The business task> Identify how consumers actually use non-Bellabeat smart devices, and determine which of the> resulting behavioural gaps Bellabeat is positioned to close through its marketing of the> Bellabeat app.### Guiding questions this report answers1. What are some trends in smart device usage?2. How could these trends apply to Bellabeat customers?3. How could these trends help influence Bellabeat marketing strategy?All three are answered in section 6.### Key stakeholders| Stakeholder | Role | What they need ||---|---|---|| Urška Sršen | Cofounder, Chief Creative Officer | Growth opportunities grounded in evidence || Sando Mur | Cofounder, executive team | Analytically sound conclusions || Marketing analytics team | Colleagues | A reproducible basis for campaign decisions |### Product selectedThe **Bellabeat app** — it connects every device in the range, is where behavioural nudgesare delivered, and drives the membership subscription, so behavioural insights can be actedon there fastest and most cheaply.

---# 2. Prepare — Description of data sources> **Deliverable 2: A description of all data sources used**| Item | Detail ||---|---|| Dataset | FitBit Fitness Tracker Data || Made available by | Mobius, via [Kaggle](https://www.kaggle.com/datasets/arashnic/fitbit) || Licence | CC0 — public domain, no restriction on use || Collection | Amazon Mechanical Turk survey, 12 March – 12 May 2016 || Consent | Participants explicitly consented to submit personal tracker data || Privacy | No names or demographics; users identified by numeric ID only |Because the data is CC0 and carries no personal identifiers, there are no licensing, privacyor security barriers to this analysis.### Files selected, and whyOf the 18 files available, six carry the analysis. `dailySteps_merged.csv`,`dailyCalories_merged.csv` and `dailyIntensities_merged.csv` were excluded as duplicates ofcolumns already in `dailyActivity_merged.csv` — verified below, not assumed.`heartrate_seconds_merged.csv` was excluded: 86 MB covering only 14 users, too small asubsample to support a finding. Minute-level activity files are redundant to the hourly anddaily aggregates.

In [ ]:
steps_only <- read_csv(file.path(P2, "dailySteps_merged.csv"), show_col_types = FALSE)activity   <- read_csv(file.path(P2, "dailyActivity_merged.csv"), show_col_types = FALSE)check <- steps_only |> rename(day = ActivityDay) |>  inner_join(activity |> rename(day = ActivityDate), by = c("Id", "day"))cat("User-days compared:", nrow(check), "\n")cat("Step counts identical in every row:", all(check$StepTotal == check$TotalSteps), "\n")

In [ ]:
display_markdown(sprintf("### Data integrity and credibility — does it ROCCC?| Criterion | Assessment | Detail ||---|---|---|| **R**eliable | Weak | %d self-selected participants; no independent verification || **O**riginal | Weak | Third-party redistribution, not collected by Fitbit or Bellabeat || **C**omprehensive | Weak | No age, gender, height, location or health baseline || **C**urrent | Poor | Collected %s — hardware and expectations have since changed || **C**ited | Good | Clearly sourced and openly licensed under CC0 |**This data does not ROCCC well, and that is stated before the findings rather than after.**The sample is small and self-selected, and it contains **no gender field** — a materiallimitation when analysing for a company selling exclusively to women. Every finding below isa directional signal to validate against Bellabeat's own data, not a conclusion to commitbudget to.",  n_users, format(min(daily$date), "%%B %%Y")))

---# 3. Process — Documentation of data cleaning> **Deliverable 3: Documentation of any cleaning or manipulation of data****Tools chosen: R with the tidyverse.** The dataset exceeds Excel's comfortable working rangeat minute level (over 1.3 million rows); every cleaning step is recorded as code and istherefore reproducible and auditable; and one environment handles cleaning, statistics andvisualisation without exporting between tools.

In [ ]:
tibble::tibble(  Check = c("Duplicate rows in daily activity",            "Duplicate rows in sleep (April roll-up)",            "Missing values in daily activity",            "Days recording zero steps",            "Weight entries with missing body-fat value"),  Result = c(sum(duplicated(daily_raw)), sum(duplicated(sleep_apr_raw)),             sum(is.na(daily_raw)), sum(!daily$is_wear_day), sum(is.na(weight$fat))))

In [ ]:
daily |>  group_by(`Day type` = ifelse(is_wear_day, "Device worn", "Zero steps recorded")) |>  summarise(Days = n(),            `Mean sedentary minutes` = round(mean(sedentary_minutes)),            `Mean calories` = round(mean(calories)), .groups = "drop")

In [ ]:
display_markdown(sprintf("**Handling non-wear days.** %d rows record zero steps alongside close to a full day ofsedentary minutes — a device left on a charger, not a participant motionless for 24 hours.These were **excluded from all activity averages** but **retained for the device-engagementanalysis**, where they are the direct evidence of how often the tracker is worn. Includingthem in averages would understate activity by roughly %s.**Extending sleep coverage into March.** `sleepDay_merged.csv` exists only in the Aprilfolder. `minuteSleep_merged.csv` covers March at minute level, where `value` is coded1 = asleep, 2 = restless, 3 = awake, and every row is one minute in bed. Counting rows pernight gives time in bed; counting `value == 1` gives minutes asleep — the same two measuresthe April file reports.*Assumption declared:* sleep crosses midnight, so grouping by calendar date would split onenight into two records. Subtracting six hours before taking the date assigns pre-dawn minutesto the previous night, and records under 60 minutes were dropped as fragments. This 6 a.m.cutoff is a judgement call.",  sum(!daily$is_wear_day), percent(mean(!daily$is_wear_day), 0.1)))

In [ ]:
tibble::tibble(  Table = c("Daily activity (worn days)", "Sleep", "Hourly steps", "Weight logs"),  Rows  = c(nrow(worn), nrow(sleep), nrow(hourly), nrow(weight)),  Users = c(n_distinct(worn$id), n_distinct(sleep$id),            n_distinct(hourly$id), n_distinct(weight$id)),  `Date range` = c(    paste(format(min(worn$date), "%d %b"), "-", format(max(worn$date), "%d %b %Y")),    paste(format(min(sleep$date), "%d %b"), "-", format(max(sleep$date), "%d %b %Y")),    paste(format(min(as_date(hourly$ts)), "%d %b"), "-", format(max(as_date(hourly$ts)), "%d %b %Y")),    paste(format(min(weight$date), "%d %b"), "-", format(max(weight$date), "%d %b %Y"))))

---# 4. Analyze — Summary of the analysis> **Deliverable 4: A summary of your analysis**The analysis proceeded in four passes. **Descriptive statistics** established baselineactivity, sleep and calorie levels against published health benchmarks. **User-levelsegmentation** grouped participants by average daily steps and by device wear rate — alwaysaggregating per user before comparing users, so participants with more logged days do notdominate the averages. **Temporal aggregation** examined activity by hour of day and day ofweek. **Correlation and regression** tested the relationships between steps, calories,sedentary time and sleep duration.**The central surprise:** the variable most strongly associated with sleep is not step countbut **sedentary time**, and the relationship is roughly three times stronger. This reframesthe marketing proposition from encouraging exercise to reducing sitting — a substantiallylower bar for the user, and a claim no major competitor is making.

In [ ]:
tibble::tibble(  Relationship = c("Daily steps vs calories burned",                   "Sedentary minutes vs calories burned",                   "Daily steps vs hours asleep",                   "Sedentary minutes vs hours asleep"),  `Pearson r` = c(round(cor(worn$total_steps, worn$calories), 3),                  round(cor(worn$sedentary_minutes, worn$calories), 3),                  round(cor(sa$total_steps, sa$hours_asleep), 3),                  round(cor(sa$sedentary_minutes, sa$hours_asleep), 3)),  n = c(nrow(worn), nrow(worn), nrow(sa), nrow(sa)))

---# 5. Share — Visualisations and key findings> **Deliverable 5: Supporting visualisations and key findings**## Finding 1 — The device comes off, and often

In [ ]:
usage <- wear |>  mutate(segment = case_when(rate >= 0.80 ~ "High use (80-100%)",                             rate >= 0.50 ~ "Moderate use (50-79%)",                             TRUE         ~ "Low use (<50%)") |>           factor(levels = c("High use (80-100%)", "Moderate use (50-79%)", "Low use (<50%)")))usage_sum <- usage |> count(segment, .drop = FALSE) |> mutate(pct = n / sum(n))ggplot(usage_sum, aes(reorder(segment, n), n, fill = segment)) +  geom_col(width = .62) +  geom_text(aes(label = paste0(n, " users (", percent(pct, 1), ")")),            hjust = -0.08, size = 4.2, colour = SLATE) +  coord_flip() + scale_y_continuous(expand = expansion(c(0, .3))) +  scale_fill_manual(values = c(TEAL, MINT, CORAL), guide = "none") +  labs(title = paste0("Median user wore the tracker on just ",                      percent(median(usage$rate), 1), " of days"),       subtitle = "Days with any steps logged, as a share of each user's export window",       x = NULL, y = "Number of users") + theme_bb

In [ ]:
display_markdown(sprintf("The median user logged steps on **%s** of available days. **No participant** wore the deviceon 80%%%% or more of days, and %d wore it on fewer than half.> **Why it matters:** a tracker spending a third of its life in a drawer cannot deliver on> sleep, stress or cycle insight. Engagement is the binding constraint on every other feature.",  percent(median(usage$rate), 1),  usage_sum$n[usage_sum$segment == "Low use (<50%)"]))

## Finding 2 — The 10,000-step goal is a wall, not a target

In [ ]:
user_avg <- worn |> group_by(id) |>  summarise(avg_steps = mean(total_steps), .groups = "drop") |>  mutate(level = case_when(avg_steps < 5000  ~ "Sedentary (<5k)",                           avg_steps < 7500  ~ "Low active (5-7.5k)",                           avg_steps < 10000 ~ "Somewhat active (7.5-10k)",                           TRUE              ~ "Active (10k+)") |>           factor(levels = c("Sedentary (<5k)", "Low active (5-7.5k)",                             "Somewhat active (7.5-10k)", "Active (10k+)")))lvl <- user_avg |> count(level, .drop = FALSE) |> mutate(pct = n / sum(n))ggplot(lvl, aes(level, n, fill = level)) +  geom_col(width = .62) +  geom_text(aes(label = paste0(n, "\n", percent(pct, 1))),            vjust = -0.25, size = 4, colour = SLATE) +  scale_y_continuous(expand = expansion(c(0, .25))) +  scale_fill_manual(values = c(CORAL, SAND, MINT, TEAL), guide = "none") +  labs(title = paste0(sum(lvl$n[1:2]), " of ", sum(lvl$n),                      " users average under 7,500 steps a day"),       subtitle = "Users grouped by their own average daily step count",       x = NULL, y = "Number of users") + theme_bb

In [ ]:
worn |> group_by(weekday) |>  summarise(avg_steps = mean(total_steps), .groups = "drop") |>  ggplot(aes(weekday, avg_steps, fill = avg_steps)) +  geom_col(width = .68) +  geom_hline(yintercept = 10000, linetype = "dashed", colour = CORAL, linewidth = .7) +  annotate("text", x = 1, y = 10400, label = "10,000-step goal",           colour = CORAL, size = 3.6, hjust = 0) +  scale_fill_gradient(low = MINT, high = TEAL, guide = "none") +  scale_y_continuous(labels = comma, expand = expansion(c(0, .12))) +  labs(title = "No day of the week reaches the 10,000-step goal on average",       subtitle = "Sunday is the least active day", x = NULL, y = "Average steps") + theme_bb

In [ ]:
display_markdown(sprintf("Mean daily steps were **%s** and the median **%s**. Only **%s** of days reached 10,000 steps.> **Why it matters:** for half the user base the default goal is unreachable. A goal missed> daily stops functioning as motivation and becomes a reminder of failure.",  comma(mean(worn$total_steps), 1), comma(median(worn$total_steps), 1),  percent(mean(worn$total_steps >= 10000), 0.1)))

## Finding 3 — The problem is sitting, not exercising

In [ ]:
options(repr.plot.height = 4)# NOTE: the value column is `avg_min`, NOT `minutes` - `minutes` is also a# lubridate function, and a failed column lookup would silently fall back to it.mins <- worn |> ungroup() |>  summarise(Sedentary = mean(sedentary_minutes), Light = mean(lightly_active_minutes),            Fair = mean(fairly_active_minutes), Very = mean(very_active_minutes)) |>  pivot_longer(everything(), names_to = "intensity", values_to = "avg_min") |>  mutate(pct = avg_min / sum(avg_min),         intensity = factor(intensity, levels = c("Sedentary", "Light", "Fair", "Very")))ggplot(mins, aes("", avg_min, fill = intensity)) +  geom_col(width = .55) +  geom_text(aes(label = ifelse(pct > .03, paste0(intensity, "\n", round(avg_min), " min"), "")),            position = position_stack(vjust = .5), size = 4,            colour = "white", fontface = "bold") +  coord_flip() + scale_fill_manual(values = c(SLATE, MINT, SAND, CORAL)) +  labs(title = paste0(percent(mins$pct[mins$intensity == "Sedentary"], 1),                      " of tracked time is sedentary"),       subtitle = "Average minutes per worn day by intensity band",       x = NULL, y = "Minutes", fill = NULL) +  theme_bb + theme(axis.text.y = element_blank())

In [ ]:
options(repr.plot.height = 5.5)ct <- cor.test(worn$total_steps, worn$calories)m  <- lm(calories ~ total_steps, data = worn)ggplot(worn, aes(total_steps, calories)) +  geom_point(alpha = .28, colour = TEAL, size = 1.5) +  geom_smooth(method = "lm", se = TRUE, colour = CORAL, fill = SAND) +  scale_x_continuous(labels = comma) + scale_y_continuous(labels = comma) +  labs(title = "More steps means more calories burned - but the payoff is modest",       subtitle = paste0("r = ", round(ct$estimate, 2), ", about ",                         round(coef(m)[2] * 1000), " calories per additional 1,000 steps"),       x = "Daily steps", y = "Calories burned") + theme_bb

In [ ]:
display_markdown(sprintf("Users averaged **%s sedentary hours** per tracked day against **%s very-active minutes**.On **%s** of days, combined fairly- and very-active time fell under 30 minutes.> **Why it matters:** the addressable opportunity is the %s sedentary hours, not the %d> active minutes. Breaking up sitting is a far lower bar than adding workouts.",  round(mean(worn$sedentary_minutes)/60, 1), round(mean(worn$very_active_minutes), 1),  percent(mean((worn$fairly_active_minutes + worn$very_active_minutes) < 30), 0.1),  round(mean(worn$sedentary_minutes)/60, 1), round(mean(worn$very_active_minutes))))

## Finding 4 — Activity clusters at two predictable peaks

In [ ]:
by_hour <- hourly |> group_by(hour) |>  summarise(avg_steps = mean(step_total), .groups = "drop")ggplot(by_hour |> mutate(peak = hour %in% c(12, 13, 17, 18, 19)),       aes(factor(hour), avg_steps, fill = peak)) +  geom_col(width = .78) +  scale_fill_manual(values = c(`FALSE` = GREY, `TRUE` = TEAL), guide = "none") +  labs(title = "Activity peaks at lunch and again from 5-7pm",       subtitle = "Average steps per hour of day, all users pooled",       x = "Hour of day", y = "Average steps") + theme_bb

In [ ]:
by_hour |> arrange(desc(avg_steps)) |> head(5) |>  transmute(Hour = paste0(hour, ":00"), `Average steps` = round(avg_steps))

> **Why it matters:** notification timing is guesswork for most apps. These windows are when> users are already in motion and most receptive to a prompt.## Finding 5 — Half of all nights fall short of healthy sleep

In [ ]:
ggplot(sleep, aes(hours_asleep)) +  geom_histogram(binwidth = .5, fill = TEAL, colour = "white") +  geom_vline(xintercept = 7, linetype = "dashed", colour = CORAL, linewidth = .8) +  annotate("text", x = 7.12, y = Inf, vjust = 2, hjust = 0, colour = CORAL, size = 3.8,           label = "7 hours = minimum recommended") +  labs(title = paste0(percent(mean(sleep$hours_asleep < 7), 1),                      " of nights fall short of 7 hours of sleep"),       subtitle = paste0(nrow(sleep), " nights from ", n_distinct(sleep$id), " users"),       x = "Hours asleep", y = "Nights") + theme_bb

In [ ]:
ggplot(sa, aes(sedentary_minutes, hours_asleep)) +  geom_point(alpha = .3, colour = TEAL, size = 1.5) +  geom_smooth(method = "lm", se = TRUE, colour = CORAL, fill = SAND) +  geom_hline(yintercept = 7, linetype = "dotted", colour = SLATE) +  labs(title = "The more sedentary the day, the less sleep that night",       subtitle = paste0("r = ", round(cor(sa$sedentary_minutes, sa$hours_asleep), 2),                         " across ", nrow(sa), " matched user-days"),       x = "Sedentary minutes", y = "Hours asleep") + theme_bb

In [ ]:
display_markdown(sprintf("Mean sleep was **%s hours**, with an average of **%s minutes** spent awake in bed.Across %d matched user-days, sedentary minutes correlate with sleep at **r = %s** — againstonly r = %s for step count.> **Why it matters:** this is the most actionable relationship in the dataset. The> proposition is not \"walk more, sleep better\" but **\"sit less, sleep better\"** — a> concrete, defensible and differentiated claim.",  round(mean(sleep$hours_asleep), 2), round(mean(sleep$awake_in_bed), 1), nrow(sa),  round(cor(sa$sedentary_minutes, sa$hours_asleep), 2),  round(cor(sa$total_steps, sa$hours_asleep), 2)))

## Finding 6 — Adoption collapses when effort is required

In [ ]:
adopt <- tibble::tibble(  feature = c("Activity (passive)", "Sleep (wear overnight)", "Weight (manual entry)"),  users   = c(n_distinct(daily$id), n_distinct(sleep$id), n_distinct(weight$id))) |> mutate(pct = users / n_users)ggplot(adopt, aes(reorder(feature, pct), pct, fill = feature)) +  geom_col(width = .6) +  geom_text(aes(label = paste0(users, " users (", percent(pct, 1), ")")),            hjust = -0.08, size = 4.2, colour = SLATE) +  coord_flip() +  scale_y_continuous(labels = percent, limits = c(0, 1.3), expand = c(0, 0)) +  scale_fill_manual(values = c(TEAL, SAND, CORAL), guide = "none") +  labs(title = "Tracking drops sharply once it needs manual input",       subtitle = "Share of users who logged each data type at least once",       x = NULL, y = "Share of users") + theme_bb

In [ ]:
display_markdown(sprintf("**%s** of weight entries were typed in by hand, with a median of **%d logs per logginguser** across the whole window.> **Why it matters:** each increment of friction costs roughly a third of the user base.> Features requiring manual input will not be adopted regardless of design quality.",  percent(mean(weight$is_manual_report == TRUE, na.rm = TRUE), 0.1),  median(count(weight, id)$n)))

In [ ]:
display_markdown(sprintf("---# 6. Act — Recommendations> **Deliverable 6: Top high-level content recommendations**## Answers to the three guiding questions**1. What are some trends in smart device usage?**Devices are worn inconsistently — a median of %s of days, with no user exceeding 80%%%%.Activity falls short of public health targets: only %s of days reach 10,000 steps, and %shours of the average tracked day are sedentary. Activity concentrates at lunchtime and earlyevening. Sleep is short — %s of nights fall under seven hours — and feature adoption dropssharply whenever manual input is required.**2. How could these trends apply to Bellabeat customers?**Bellabeat customers use the same categories of device for the same purposes, so the samebehavioural ceilings apply. First, wear rate caps everything: Bellabeat's stress, sleep andcycle features depend on consistent wear, so the Leaf's jewellery form factor addresses areal constraint rather than a cosmetic one. Second, the sedentary-sleep link maps preciselyonto Bellabeat's positioning around holistic wellness rather than athletic performance.**3. How could these trends help influence Bellabeat marketing strategy?**They shift the message from performance to feasibility. Rather than competing with Fitbit andApple on step counts and workout tracking — where the data shows most users are failing —Bellabeat can own the lower, more achievable and better evidenced proposition of sitting lessto sleep better.## Top three recommendations### 1. Replace the fixed step goal with an adaptive oneOnly %s of days reach 10,000 steps and %d of %d users average under 7,500. Set each user'sinitial target from their own first-week baseline and raise it incrementally.*Marketing line:* **\"A goal that meets you where you are.\"**### 2. Market the app on the sit-less, sleep-better linkSedentary time predicts short sleep roughly three times more strongly than step count does(r = %s against r = %s). Build the flagship app feature around this relationship, pairinghourly move reminders with the sleep score.*Marketing line:* **\"Sit less today, sleep better tonight.\"**### 3. Time notifications to the two peaksActivity peaks at 12-2 p.m. and 5-7 p.m. Deliver movement prompts in those windows, the dailyplan in the 7 a.m. lull, and a wind-down prompt around 10 p.m. This is the cheapest of thethree to implement and the fastest to measure.*Marketing line:* **\"Nudges when you're already moving.\"**## Supporting recommendations**4. Make the Leaf's form factor the retention pitch.** Wear rate is the ceiling on everyother feature. Market the Leaf as *\"the tracker you don't take off\"*, and rewardconsecutive days **worn** rather than steps achieved.**5. Eliminate manual logging.** Weight logging reached %s of users. Assume any hand-entryfeature will fail; prioritise Spring's automatic hydration sync and smart-scale integration.**6. Position membership content around midweek.** Sleep and activity both dip midweek.Schedule coaching content for Sunday evening and Tuesday morning.",  percent(median(usage$rate), 1),  percent(mean(worn$total_steps >= 10000), 0.1),  round(mean(worn$sedentary_minutes)/60, 1),  percent(mean(sleep$hours_asleep < 7), 0.1),  percent(mean(worn$total_steps >= 10000), 0.1),  sum(lvl$n[1:2]), n_users,  round(cor(sa$sedentary_minutes, sa$hours_asleep), 2),  round(cor(sa$total_steps, sa$hours_asleep), 2),  percent(adopt$pct[3], 1)))

---# 7. Limitations and next steps

In [ ]:
tibble::tibble(  Limitation = c("Sample size", "Gender data", "Data age", "Recruitment",                 "Causality", "Sleep aggregation"),  Detail = c(    paste0(n_users, " participants - too few to generalise to a consumer market"),    "Not recorded, yet Bellabeat sells exclusively to women",    paste0("Collected ", format(min(daily$date), "%b %Y"), " to ",           format(max(daily$date), "%b %Y"), " - nine years old"),    "Self-selected MTurk volunteers, unlikely to match Bellabeat's customer base",    "The sedentary-sleep link is an association; direction is untested",    "March nights rely on a 6 a.m. cutoff assumption"))

**These findings should be validated before budget is committed.** Recommended next steps,in order of cost:1. **A/B test notification timing** against the 12–2 p.m. and 5–7 p.m. windows. Cheapest to   run and fastest to prove or disprove.2. **Replicate this analysis on Bellabeat's own app telemetry**, where gender, age and   product line are known — the additional dataset Sršen suggested.3. **Test the adaptive goal** against the fixed 10,000-step goal, measuring 30-day retention   rather than step count.4. **Survey lapsed users** on why the device came off, to confirm whether form factor is   genuinely the binding constraint on wear rate.

In [ ]:
sessionInfo()